In [ ]:
!pip install google-adk -q
!pip install litellm -q

In [ ]:
import os
import requests
from typing import Dict, Optional
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from vertexai.preview.reasoning_engines import AdkApp
import asyncio
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types # For creating response content
from typing import Optional
import logging
from typing import Optional
import sys
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.adk.agents import Agent
from google.genai.types import Content, Part
from vertexai.preview.reasoning_engines import AdkApp
from google.adk.tools import google_search

In [ ]:
AGENT_MODEL = "gemini-2.5-flash"
API_KEY = ""
llm_model = LiteLlm(model=AGENT_MODEL, api_key=API_KEY)

In [ ]:
import requests
from typing import Dict, Any, Optional

def get_weather_forecast(latitude: float, longitude: float) -> Optional[Dict[str, Any]]:
    """Retrieves the weather forecast from the National Weather Service (NWS) API.

    This function takes a latitude and longitude, first determines the
    appropriate NWS grid endpoint for that location, and then fetches the
    detailed weather forecast.

    Args:
        latitude: The latitude of the location (e.g., 38.8951).
        longitude: The longitude of the location (e.g., -77.0364).

    Returns:
        A dictionary containing the forecast properties, which includes a
        list of forecast periods. Returns None if the forecast could
        not be retrieved due to an API error or network issue.
    """
    # The NWS API requires a unique User-Agent header.
    # Replace "(My Weather App, myemail@example.com)" with your app's details.
    headers = {
        'User-Agent': '(My Weather App, myemail@example.com)'
    }

    # Step 1: Get the gridpoint metadata URL from the initial coordinates.
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"

    try:
        # First API call to get the specific forecast URL
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

        # Extract the URL for the actual forecast from the response
        forecast_url = points_response.json()['properties']['forecast']

        # Step 2: Get the actual forecast data from the forecast URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Return the 'properties' key which contains the forecast data
        return forecast_response.json()['properties']

    except requests.exceptions.RequestException as e:
        print(f"An error occurred while communicating with the NWS API: {e}")
        return None
    except (KeyError, ValueError) as e:
        # Handle cases where the JSON response is not as expected
        print(f"Error parsing the API response: {e}")
        return None

In [ ]:
!pip install googlemaps -q

In [ ]:
# logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

In [ ]:
# -- user moderation -- #

USE_MODEL_ARMOR = True

def check_user_input_model_armor(user_text: str) -> str:
    """
    Sends user input to Google Model Armor for content moderation.
    Returns 'BAD' if content is unsafe, 'GOOD' if safe.
    """
    try:
        token = get_fresh_token()

        url = f"https://modelarmor.us.rep.googleapis.com/v1/{TEMPLATE_NAME}:sanitizeUserPrompt"

        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        }
        payload = {"userPromptData": {"text": user_text}}

        response = requests.post(url, json=payload, headers=headers, timeout=10)
        print("Status:", response.status_code)
        print("Response:", response.text)

        result = response.json()

        match_state = (
            result
            .get("sanitizationResult", {})
            .get("filterMatchState", "NO_MATCH_FOUND")
        )
        return "BAD" if match_state == "MATCH_FOUND" else "GOOD"

    except Exception as e:
        logger.warning("Model Armor unavailable (%s), sending BAD", e)
        return "BAD"


def moderate_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """
    Checks the latest user message for content policy violation.
    Returns a blocking response if content is inappropriate,
    or None to allow the request to proceed normally.
    """
    try:
        last_user_message = None
        for message in reversed(llm_request.contents):
            if message.role == "user":
                last_user_message = message
                break

        if not last_user_message or not last_user_message.parts:
            return None
        part = last_user_message.parts[0]
        if not hasattr(part, "text") or part.text is None: # tool call
            return None

        user_text = part.text.strip()
        if not user_text:
            return None

        logger.info("[%s] MODERATING » %s", callback_context.agent_name, user_text[:100])
        result = check_user_input_model_armor(user_text)

        if result.strip().upper() == "BAD":
            logger.warning("[%s] BLOCKED inappropriate content", callback_context.agent_name)
            return LlmResponse(
                content=types.Content(
                    role="model",
                    parts=[types.Part(
                        text="Your message violates our content guidelines. "
                             "Please rephrase your request."
                    )]
                )
            )

    except Exception as e:
        logger.exception("Moderation callback failed: %s", e)

    return None

In [ ]:
# -- input logging -- #

def log_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """
    Logs the user's input for auditing purposes.
    Always returns None so the request proceeds normally.
    """
    try:
        for message in reversed(llm_request.contents):
            if message.role == "user" and message.parts:
                part = message.parts[0]
                if not hasattr(part, "text") or part.text is None:
                    continue
                user_text = part.text.strip()
                if user_text:
                    logger.info("[%s] USER INPUT » %s", callback_context.agent_name, user_text[:200])
                    break
    except Exception as e:
        logger.exception("Log user prompt callback failed: %s", e)

    return None

In [ ]:
# -- model response logger - callback after model -- #

def log_model_response(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
    """
    Writes the first text part of the model's response to the log
    and passes the response through unchanged.

    Args:
        callback_context: Contains agent name, session info, state.
        llm_response: The raw response from the LLM.

    Returns:
        None to pass the response through unchanged.
    """
    agent_name = callback_context.agent_name

    if llm_response.content and llm_response.content.parts:
        # Check if this is a text response (vs a tool call)
        first_part = llm_response.content.parts[0]

        if hasattr(first_part, "text") and first_part.text:
            txt = first_part.text.strip()
            logger.info("[%s] MODEL RESPONSE » %s", agent_name, txt[:200])
        else:
            # It's a tool call, not text
            logger.info("[%s] MODEL » (tool call initiated)", agent_name)
    else:
        logger.info("[%s] MODEL » (empty response)", agent_name)

    return None

In [ ]:
import googlemaps
from typing import Optional, Tuple

# -- GET LAT LON TOOL -- #

def get_lat_lon(location: str) -> Optional[Dict[str, float]]:
    """Converts a physical address to latitude and longitude using Google's Geocoding API.

    Args:
        api_key: Your Google Maps API key with the Geocoding API enabled.
        address: The human-readable address or place name to geocode.

    Returns:
        A tuple containing the latitude and longitude, or None if an error occurs.
    """
    # Initialize the client with your API key
    gmaps = googlemaps.Client(key=API_KEY)

    try:
        # Make the API call to geocode the address
        geocode_result = gmaps.geocode(location)

        if not geocode_result:
            print(f"Warning: No results found for address: '{location}'")
            return None

        location = geocode_result[0]['geometry']['location']
        latitude = location['lat']
        longitude = location['lng']

        return (latitude, longitude)

    except googlemaps.exceptions.ApiError as e:
        print(f"An API error occurred: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

print(get_lat_lon("New York, NY"))

(40.7127753, -74.0059728)


In [ ]:
# -- authentication for model armor -- #

from google.colab import auth
auth.authenticate_user()

def get_fresh_token() -> str:
    """Gets a fresh GCP access token."""
    return subprocess.check_output(
        ["gcloud", "auth", "print-access-token"],
        text=True
    ).strip()

# Get your project ID
import subprocess
PROJECT_ID = subprocess.check_output(
    ["gcloud", "config", "get-value", "project"],
    text=True
).strip()

# Get access token
ACCESS_TOKEN = subprocess.check_output(
    ["gcloud", "auth", "print-access-token"],
    text=True
).strip()

print(f"Project: {PROJECT_ID}")
print(f"Token: {ACCESS_TOKEN[:20]}...")

LOCATION = "us"  # ← Must match your template's location exactly

TEMPLATE_NAME = "projects/qwiklabs-gcp-00-819951dd6b01/locations/us/templates/USModelArmor"

url = f"https://modelarmor.{LOCATION}.rep.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/templates"
headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

response = requests.get(url, headers=headers)
templates = response.json()

for t in templates.get("modelArmorTemplates", []):
    print("Template name:", t["name"])

Project: qwiklabs-gcp-00-819951dd6b01
Token: ya29.a0ATkoCc4zBP497...


In [ ]:
def chained_before_callback(
    callback_context: CallbackContext,
    llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """
    Chains multiple before_model callbacks:
    1. Moderate user input (can block)
    2. Log user input (observes only)

    Returns blocking response if moderation fails, else None.
    """
    # Step 1: Moderation — if it returns something, STOP immediately
    moderation_result = moderate_user_prompt(callback_context, llm_request)
    if moderation_result is not None:
        return moderation_result  # ← Blocked. LLM never called.

    # Step 2: Logging — always returns None, just for audit trail
    log_user_prompt(callback_context, llm_request)

    return None

In [ ]:
# This cell previously contained a custom google_search function, which is no longer needed.
# We will use the built-in GoogleSearch tool from ADK instead. This cell can now be removed or left empty.

In [ ]:
# -- AGENTS -- #

weather_agent = Agent(
    name="weather_agent_v1",
    model=llm_model, # Can be a string for Gemini or a LiteLlm object
    description="Provides weather information for specific cities.",
    instruction="""
    You are a helpful weather assistant for users in the USA.
    When a user asks for the forecast for a specific city:
    1. use get_lat_lon to convert the name of the city into necessary coordinates.
    2. use get_weather_forecast to get the weather forecast for the city.
    3. format the results:
      - Forecast for today including highs lows and precipitation
      - Recommendations about what to wear or whether you need an umbrella
      - Additional forecast
    """,
    tools=[get_lat_lon, get_weather_forecast], # Pass the function directly
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response
)

search_agent = Agent(
    name="search_agent_v1",
    model=llm_model, # Can be a string for Gemini or a LiteLlm object
    description="Agent specializing in information lookup",
    instruction="""
    You are a helpful research specialist.
    STRICT RULES:
    - ALWAYS use the 'google_search' tool for information retrieval.
    - NEVER delegate or transfer control to another agent.
    Use the 'google_search' tool to find information that is the most relevant to the user's request.
    Return findings in a concise and clear manner.
    """,
    tools=[google_search], # Using the built-in google_search function
)

root_agent = Agent(
    name="root_agent",
    model=llm_model,
    description="Primary agent for routing user requests to relevant subagents",
    instruction="""
    You are a helpful assistant that coordinates subagents.

    STRICT RULES:
    - For ANY weather/forecast/alerts question → ALWAYS delegate to "weather_agent_v1"
    - For ANY research/facts/history question → ALWAYS delegate to "search_agent_v1"
    - If a question has BOTH weather AND research components → delegate to BOTH
    """,
    # sub_agents: root delegates to these agents automatically
    sub_agents=[weather_agent, search_agent],
)

In [ ]:
from google.colab import auth
auth.authenticate_user()

# Get your project ID
import subprocess
PROJECT_ID = subprocess.check_output(
    ["gcloud", "config", "get-value", "project"],
    text=True
).strip()

# Get access token
ACCESS_TOKEN = subprocess.check_output(
    ["gcloud", "auth", "print-access-token"],
    text=True
).strip()

print(f"Project: {PROJECT_ID}")
print(f"Token: {ACCESS_TOKEN[:20]}...")

LOCATION = "us"  # ← Must match your template's location exactly

TEMPLATE_NAME = "projects/qwiklabs-gcp-00-819951dd6b01/locations/us/templates/USModelArmor"

url = f"https://modelarmor.{LOCATION}.rep.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/templates"
headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

response = requests.get(url, headers=headers)
templates = response.json()

for t in templates.get("modelArmorTemplates", []):
    print("Template name:", t["name"])

Project: qwiklabs-gcp-00-819951dd6b01
Token: ya29.a0ATkoCc4zQArfO...


In [ ]:
async def ask_root(question: str, session_id: str = "test_session") -> str:
    """Run a single question through the weather agent and return the response."""
    session_service = InMemorySessionService()

    runner = Runner(
        agent=root_agent,
        app_name="subagent_app",
        session_service=session_service,
    )

    session = await session_service.create_session(
        app_name="subagent_app",
        user_id="test_user",
        session_id=session_id,
    )

    response_text = ""
    async for event in runner.run_async(
        user_id="test_user",
        session_id=session.id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=question)]
        ),
    ):
        if event.is_final_response():
            if event.content and event.content.parts and event.content.parts[0].text:
                response_text = event.content.parts[0].text
            else:
                response_text = "(No response received — likely a rate limit or empty reply)"

    return response_text

In [ ]:
async def run_ch3_tests():
    tests = [
        ("weather", "What's the weather forecast for New York, NY?"),
        ("search",  "What is the history of the National Weather Service?"),
        ("both",    "What US city had the most recent blizzard, and what is the weather like there now?"),
    ]

    for test_type, question in tests:
        print(f"\n🧪 [{test_type.upper()}] {question}")
        print("-" * 55)

        # Retry logic for 503 Service Unavailable errors
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = await ask_root(question, session_id=f"ch3_{test_type}")
                print(response)
                break # Success, exit retry loop
            except Exception as e:
                # Check for 503 error string
                if "503" in str(e) or "UNAVAILABLE" in str(e):
                    if attempt < max_retries - 1:
                        wait_time = (attempt + 1) * 10
                        print(f"⚠️ Model overloaded (503). Retrying in {wait_time}s...")
                        await asyncio.sleep(wait_time)
                    else:
                         print(f"❌ Failed after {max_retries} attempts: {e}")
                else:
                    print(f"❌ Error: {e}")
                    break # Not a 503, stop trying

        print()
        await asyncio.sleep(15)  # Rate limit buffer

await run_ch3_tests()


🧪 [WEATHER] What's the weather forecast for New York, NY?
-------------------------------------------------------
Status: 200
Response: {
  "sanitizationResult": {
    "filterMatchState": "NO_MATCH_FOUND",
    "filterResults": {
      "csam": {
        "csamFilterFilterResult": {
          "executionState": "EXECUTION_SUCCESS",
          "matchState": "NO_MATCH_FOUND"
        }
      },
      "rai": {
        "raiFilterResult": {
          "executionState": "EXECUTION_SUCCESS",
          "matchState": "NO_MATCH_FOUND",
          "raiFilterTypeResults": {
            "sexually_explicit": {
              "matchState": "NO_MATCH_FOUND"
            },
            "hate_speech": {
              "matchState": "NO_MATCH_FOUND"
            },
            "harassment": {
              "matchState": "NO_MATCH_FOUND"
            },
            "dangerous": {
              "matchState": "NO_MATCH_FOUND"
            }
          }
        }
      },
      "pi_and_jailbreak": {
        "piAndJailbre

In [ ]:
async def ask_search(question: str, session_id: str = "test_session_search") -> str:
    """Run a single search question through the search agent and return the response."""
    session_service = InMemorySessionService()

    runner = Runner(
        agent=search_agent,  # Use search_agent directly
        app_name="search_app",
        session_service=session_service,
    )

    session = await session_service.create_session(
        app_name="search_app",
        user_id="test_user_search",
        session_id=session_id,
    )

    response_text = ""
    async for event in runner.run_async(
        user_id="test_user_search",
        session_id=session.id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=question)]
        ),
    ):
        # --- Temporarily print all events for debugging --- #
        print(f"[DEBUG EVENT] {event}") # Modified to print the event object directly
        # -------------------------------------------------- #

        if event.is_final_response():
            if event.content and event.content.parts and event.content.parts[0].text:
                response_text = event.content.parts[0].text
            else:
                response_text = "(No response received \u2014 likely a rate limit or empty reply)"

    return response_text

In [ ]:
async def run_search_tests():
    print("=" * 60)
    print("SEARCH AGENT TESTS")
    print("=" * 60)

    tests = [
        ("Who is the most recent winner of an NBA match, and when did it take place?")
    ]

    for question in tests:
        print(f"\n\u2756 SEARCH TEST: {question}")
        print("-" * 55)

        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = await ask_search(question, session_id=f"search_test_{hash(question)}")
                print(response)
                break
            except Exception as e:
                if "503" in str(e) or "UNAVAILABLE" in str(e):
                    if attempt < max_retries - 1:
                        wait_time = (attempt + 1) * 10
                        print(f"\u26a0\ufe0f Model overloaded (503). Retrying in {wait_time}s...")
                        await asyncio.sleep(wait_time)
                    else:
                         print(f"\u274c Failed after {max_retries} attempts: {e}")
                else:
                    print(f"\u274c Error: {e}")
                    break

        print()
        await asyncio.sleep(5) # Rate limit buffer

await run_search_tests()

SEARCH AGENT TESTS

❖ SEARCH TEST: Who is the most recent winner of an NBA match, and when did it take place?
-------------------------------------------------------
[DEBUG EVENT] model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'query': 'most recent winner of an NBA match'
        },
        id='call_6c63e7a00a4c4ce0ad8f8c2b7ad1__thought__CowFAY89a18MggVGouyag0QgKPILxhRqb2UH2AmJK78ZYU+yDqmErXjSpu9Vnruf7r0bwsIi4TfDvHoRyX5P2wA8OfAYRhfMjAK5yVw5xgyRmVONl9WxkKDfYyosZBj9ax4inYwvuFgx20EkIqnODPZWdSzzldnnxgb85PHAPrO/TIAqNJ/eezo49eBkGKDWxOmQbmg2dwA9U7olSC1DmXQt4eppInX2P9yHNh4JXYxAK0NT4joA4GSR6SMJM1uuI+yTcJiBvxVNafCqWTmsOqT5zW/dpuf7CkphZVdgI4Nr5akh1LFp0iID0KG6iFnKGIWN//DWOSQ47tv30MvRGCKMriYJAqfdG18VjGNZDPyEwnnKBfGJdVArLbMDCOJBVESnCz/4T1bQcJw+5GuXYHX7zFBkWSvD/jr0qL/DXHaZCO+RW8y4qdU17UIl2+Y8STygiEngsUUpdnXOFFszkTivDhg1ss1M1n1jlX+YL7qfNjZqFhr+XWBkaCTzNZZOUX/k0h+hvMgjCdbntkXA6VxAGbL0EJmzT3fAUNHqPLAxw+lCWXihzztrXdqg/u/66F